# 15.b — Ad hoc test: the heat-unit floor of 15.a on the lower part of the cell (lapse-rate correction)

**Ad hoc test, outside the suitability chain**: it reads the heat units of 15.a (cell mean), the daily ERA5 temperatures
(as 15.a), the calendar and BIO variables of 1.1.d, the delta BIO variables of the runs (1.3.a), the relief layer of
0.5 and the final suitability of 6.b, and writes only in `<out_root>/test_heat_sum_floor_lapse/<route>/<set>/`. Nothing
of the chain, nor the outputs of 15.a, is changed.

**Question.** In 15.a the GAEZ v4 heat-unit floor removes 1–11 % of today's cultivated area on ERA5 (wheat 10 %, winter
wheat 11 %, maize 5 %, potato 5 %), mostly in the Andes, at the edge of the Tibetan Plateau and in the far north. In
mountain cells the ERA5 temperature is that of the mean elevation of the 0.7° cell, while the crops are grown in the
valleys, below it. The ECOCROP TMIN test of 4.g already accounts for this (`cu.tmin_warming`, since 6 Nov 2026). Does
the same correction remove the cost of the floor in the mountains, and does the effect of the AMOC weakening found in
15.a (maize 2050 −2.4 → −5.2 %, north of 50° N −8 → −26 %) hold?

**Correction (as the TMIN test).** ΔT_z = 0.0065 °C m⁻¹ × (mean − 10th percentile of the elevation of the land pixels
of the cell), ≥ 0 (relief layer of notebook 0.5): the warming of the lowest tenth of the cell against its mean. It is
added to the **daily** mean temperature of every scenario before the heat units are counted, so that it also changes
the days above the base. Requirement, seasons, crop types and the rule (at least one crop type passes) as 15.a. In flat
cells ΔT_z ≈ 0 and 15.b reproduces 15.a (checked).

**Simpler alternative, compared.** The floor of 15.a on the cell mean, but not applied in the relief cells
(ΔT_z ≥ 1 °C, i.e. mean − 10th percentile of the elevation ≥ about 150 m), where it risks removing valley crops: no
heat units recomputed, but no heat check at all in the mountains. Three versions of the floor are therefore compared:
cell mean (15.a), lower part of the cell (15.b), cell mean not applied in relief cells.

**Not addressed.** Silage maize (northern Europe, Russia) is harvested before grain maturity; GAEZ gives it separate
requirements. 15.b changes only the elevation.

**Outputs.** (1) Cost on ERA5 with the floor on the cell mean (15.a) and on the lower part (15.b), split between relief
cells (ΔT_z ≥ 1 °C) and the rest, and north of 50° N. (2) Suitable land per scenario: 6.b, with each floor. (3) Effect
of the AMOC weakening (hosing − no hosing, cells assessable in both, as 6.b) for the three versions, all land and per
region. (4) Maps: ΔT_z and the cultivated cells recovered by each version; AMOC effect.

Outputs in `<out_root>/test_heat_sum_floor_lapse/<route>/<set>/`: `heat_units_lapse_<crop>.nc`, `cost_era5_lapse.csv`,
`areas_floor_lapse.csv`, `amoc_effect_floor_lapse.csv`, `amoc_effect_regions_floor_lapse.csv`, the figures.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: as 15.a
CLIMATE = os.environ.get("HEAT_CLIMATE", "era5")
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")
CELL_AREA_THR = 0.0005
CROPS = os.environ.get("HEAT_CROPS", "Maize_rf,Potato_rf,Soybean_rf,Rice_rf,Wheat_rf,Wheat-winter_rf,Wheat-spring_rf").split(",")
ERA5_NAME, *ERA5_PERIOD = os.environ.get("HEAT_ERA5", "ERA5,1981,2014").split(",")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]
HU_DEF = "gaez"     # sum of the daily mean temperature on the days above the base (GAEZ v4 heat units), as 15.a
RELIEF_DT = 1.0     # degC: relief cell when the lower part is at least this much warmer than the mean (about 150 m)

# ---- GAEZ v4 model documentation, table A4-6.1 (Fischer et al. 2021), as 15.a (checked against the attributes of its files)
MAIZE = {"maize temperate and subtropical": (10.0, {90: 1750, 105: 1875, 120: 2000, 135: 2125, 150: 2250, 165: 2375, 180: 2500}),
         "maize tropical lowland": (15.0, {90: 2100, 105: 2250, 120: 2400, 135: 2550}),
         "maize tropical highland": (10.0, {120: 2000, 150: 2250, 180: 2500, 210: 2750, 240: 3000, 270: 3250, 300: 3500})}
SOY = {"soybean temperate and subtropical": (10.0, {105: 1850, 120: 2000, 135: 2150, 150: 2300}),
       "soybean tropical and subtropical": (12.5, {105: 2200, 120: 2450, 135: 2700})}
RICE = {"japonica wetland rice": (7.5, {105: 1900, 120: 2100, 135: 2300, 150: 2500, 165: 2600, 180: 2700, 195: 2750}),
        "indica wetland rice": (15.0, {105: 2500, 120: 2700, 135: 2900, 150: 3000}),
        "indica dryland rice": (15.0, {105: 2300, 120: 2500, 135: 2700})}
POTATO = {"white potato": (5.0, {90: 1200, 105: 1350, 120: 1500, 135: 1650, 150: 1750, 165: 1950, 180: 2100})}
SPRING = {"spring wheat": (5.0, {90: 1200, 105: 1300, 120: 1400, 135: 1500, 150: 1600}),
          "wheat subtropical cultivars": (5.0, {105: 1400, 120: 1500, 135: 1600, 150: 1700}),
          "wheat tropical highland": (5.0, {100: 1250, 115: 1375, 130: 1500, 145: 1625, 160: 1625, 175: 1625, 190: 1625})}
WINTER = {"winter wheat": (5.0, {140: 1300, 160: 1400, 180: 1500, 200: 1600})}   # cycle 35+105 ... 50+150 days
REQ = {"Maize_rf": MAIZE, "Soybean_rf": SOY, "Rice_rf": RICE, "Potato_rf": POTATO, "Wheat-spring_rf": SPRING,
       "Wheat-winter_rf": WINTER, "Wheat_rf": {**SPRING, **WINTER}}

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = Path(os.environ.get("HEAT_DELTA") or cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET))
B6 = Path(os.environ.get("HEAT_6B") or cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET))
A15 = Path(os.environ.get("HEAT_15A") or cu.out_dir(CFG, "test_heat_sum_floor", ROUTE, SET))      # heat units of 15.a
RELIEF = Path(os.environ.get("HEAT_RELIEF") or cu.out_dir(CFG, cu.MASKS) / "relief_ecearth3.nc")  # relief layer of 0.5
OUT = Path(os.environ.get("HEAT_OUT") or cu.out_dir(CFG, "test_heat_sum_floor_lapse", ROUTE, SET))   # the only folder written
OUT.mkdir(parents=True, exist_ok=True)
SCEN = ["ERA5", *RUNS]
VERS = ["cell mean (15.a)", "lower part (15.b)", "cell mean, not in relief cells"]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
NC = I.size
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b and 15.a
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
mha = lambda a: float(np.nansum(LAND_MHA * a))

# ---- warming of the lower part of the cell against its mean, as cu.tmin_warming (read here from RELIEF)
with xr.open_dataset(RELIEF) as ds:
    DZ_GRID = cu.LAPSE_RATE * np.clip(np.nan_to_num((ds["elev_mean"] - ds["elev_p10"]).values), 0.0, None)
DZ = DZ_GRID[I, J]
RELIEF_CELL = LAND & (DZ_GRID >= RELIEF_DT)
print(f"{CLIMATE} {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]} | {NC} land cells | crops {CROPS} | heat units of 15.a: {A15} | "
      f"written only in {OUT}")
print(f"warming of the lower part (degC): median {np.median(DZ):.2f}, 90th percentile {np.percentile(DZ, 90):.2f}, "
      f"max {DZ.max():.1f}; relief cells (>= {RELIEF_DT:g} degC): {RELIEF_CELL.sum()} ({100 * RELIEF_CELL.sum() / NC:.1f} % of land cells)")

## 1. Daily mean temperature at the land cells (as 15.a), only if heat units are to be computed

In [ ]:
def hu_file(crop):
    return OUT / f"heat_units_lapse_{crop}.nc"


def hu_attrs(crop):
    return {"crop": crop, "climate": f"{CLIMATE} {ERA5_PERIOD}", "definition": HU_DEF, "requirement": str(REQ[crop]),
            "delta": str(DELTA), "relief": str(RELIEF), "lapse_rate": str(cu.LAPSE_RATE)}


def reusable(crop):
    if not hu_file(crop).exists():
        return False
    with xr.open_dataset(hu_file(crop)) as ds:
        return all(str(ds.attrs.get(k)) == str(v) for k, v in hu_attrs(crop).items())


for crop in CROPS:                                                   # the cell-mean heat units come from 15.a
    f = A15 / f"heat_units_{crop}.nc"
    assert f.exists(), f"run 15.a first: {f} missing"
    with xr.open_dataset(f) as ds:
        assert ds.attrs["requirement"] == str(REQ[crop]) and ds.attrs["definition"] == HU_DEF, f"15.a differs: {f}"
TODO = [c for c in CROPS if not reusable(c)]
print(f"heat units (lower part) read back: {[c for c in CROPS if c not in TODO]} | to compute: {TODO}")
if TODO:
    t0 = time.time()
    tx = cu.land_series(cu.read_climate(CFG, CLIMATE, "tasmax", GRID, ERA5_PERIOD), (I, J))
    da = cu.read_climate(CFG, CLIMATE, "tasmin", GRID, ERA5_PERIOD)
    TM = (tx + cu.land_series(da, (I, J))) / 2                     # daily mean temperature (days, land cells)
    del tx
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    assert TM.shape[0] == dates.size
    print(f"{dates[0].date()} -> {dates[-1].date()}, {dates.size} days, read in {(time.time() - t0) / 60:.1f} min")

## 2. Heat units of the lower part of the cell; the floor on the cell mean (15.a), on the lower part, and on the cell mean but not in relief cells

As `heat_units` of 15.a, with ΔT_z added to the daily mean temperature of every scenario. Check: in the cells without
relief (ΔT_z = 0) the heat units equal those of 15.a (up to a day at exactly the base, counted or not depending on the
float rounding: at most 0.1 % of these cells). The third version passes every relief cell.

In [ ]:
def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def heat_units(sow, ph2, mat, dT, bases, dz):
    # as 15.a (GAEZ definition only): mean over the harvest years of the heat units of the season, per (scenario, base);
    # dT: {scenario: (change of phase 1, change of phase 2)} per land cell; dz: warming of the lower part of the cell
    yr, doy = dates.year.values, dates.dayofyear.values
    years = set(yr)
    valid = np.isfinite(sow) & np.isfinite(ph2) & np.isfinite(mat)
    wrap = valid & (mat > 365)
    s0, a0, m0 = (np.nan_to_num(x) for x in (sow, ph2, mat))
    acc, cnt = {(sc, b): np.zeros(NC) for sc in dT for b in bases}, np.zeros(NC)
    for hy in range(int(ERA5_PERIOD[0]) + 1, int(ERA5_PERIOD[1]) + 1):
        if hy not in years:
            continue
        ok = valid & (((hy - 1) in years) | ~wrap)
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = s0[None] + shift, a0[None] + shift, m0[None] + shift
        p1, p2 = (T >= s) & (T < a), (T >= a) & (T <= m)
        tm = TM[idx] + dz[None]
        for sc, (d1, d2) in dT.items():
            ts = np.where(p1, tm + d1[None], np.where(p2, tm + d2[None], np.nan))
            for b in bases:
                v = np.where(ts > b, ts, 0.0)
                acc[(sc, b)] += np.where(ok, np.nansum(np.where(np.isfinite(ts), v, 0.0), axis=0), 0.0)
        cnt += ok
    return {k: np.where(cnt > 0, acc[k] / np.maximum(cnt, 1), np.nan) for k in acc}


def requirement(fam, L):
    # lower boundary of the heat units of the crop-type row with the cycle length nearest to L (clipped to the table)
    cyc = np.array(sorted(fam))
    req = np.array([fam[c] for c in cyc], float)
    k = np.abs(np.nan_to_num(L)[:, None] - cyc[None]).argmin(axis=1)
    return np.where(np.isfinite(L), req[k], np.nan)


def to_grid(x, fill=np.nan):
    g = np.full(LAND.shape, fill, np.float32 if np.asarray(x).dtype.kind == "f" else np.asarray(x).dtype)
    g[I, J] = x
    return g


def read_hu(path):
    with xr.open_dataset(path) as ds:
        H = {}
        for v in ds.data_vars:
            sc, rest = v[3:].rsplit("_base", 1)
            b, df = rest.split("_")
            if df == HU_DEF:
                H[(sc, float(b))] = ds[v].values[I, J]
    return H


FLOOR, MARGIN, CHECK = {}, {}, []
flat = DZ == 0
for crop in CROPS:
    t1 = time.time()
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        sow, ph2, mat = (ds[v].values[I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
        e1 = ((ds["BIO01"] + ds["BIO02"]) / 2).values[I, J]
        e2 = ((ds["BIO06"] + ds["BIO07"]) / 2).values[I, J]
    L = mat - sow + 1                                              # season length (days), maturity > 365 when it wraps
    fams = REQ[crop]
    bases = sorted({b for b, _ in fams.values()})
    if crop in TODO:
        dT = {"ERA5": (np.zeros(NC), np.zeros(NC))}
        for run in RUNS:
            with xr.open_dataset(DELTA / f"BIOvariables_delta_{run}_{crop}.nc") as d:
                r1 = ((d["BIO01"] + d["BIO02"]) / 2).values[I, J]
                r2 = ((d["BIO06"] + d["BIO07"]) / 2).values[I, J]
            dT[run] = (np.nan_to_num(r1 - e1), np.nan_to_num(r2 - e2))
        H = heat_units(sow, ph2, mat, dT, bases, DZ)
        xr.Dataset({f"hu_{sc}_base{b:g}_{HU_DEF}": (("lat", "lon"), to_grid(v)) for (sc, b), v in H.items()},
                   coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                   attrs={k: str(v) for k, v in hu_attrs(crop).items()}).to_netcdf(hu_file(crop))
    HV = {VERS[0]: read_hu(A15 / f"heat_units_{crop}.nc"), VERS[1]: read_hu(hu_file(crop))}
    for (sc, b), v in HV[VERS[0]].items():                         # cells without relief reproduce 15.a
        d = np.abs(HV[VERS[1]][(sc, b)] - v)[flat]
        CHECK.append({"crop": crop, "scenario": sc, "base": b, "cells without relief differing from 15.a (%)":
                      100 * np.mean(d > 0.5) if d.size else 0.0})
    # floor: at least one crop type with heat units >= its requirement; margin = best (heat units - requirement)
    req = {f: requirement(tab, L) for f, (b, tab) in fams.items()}
    for ver, H in HV.items():
        for sc in SCEN:
            mg = np.nanmax(np.stack([H[(sc, b)] - req[f] for f, (b, _) in fams.items()]), axis=0)
            MARGIN[(crop, sc, ver)] = to_grid(mg)
            FLOOR[(crop, sc, ver)] = to_grid(~(mg < 0), fill=False) & LAND   # NaN (no season) not removed by the floor
    for sc in SCEN:
        FLOOR[(crop, sc, VERS[2])] = FLOOR[(crop, sc, VERS[0])] | RELIEF_CELL
    print(f"{crop}: {(time.time() - t1) / 60:.1f} min", flush=True)
CHECK = pd.DataFrame(CHECK)
print(f"check: cells without relief whose heat units differ from 15.a by > 0.5 degC day: at most {CHECK.iloc[:, -1].max():.3f} %")
assert CHECK.iloc[:, -1].max() < 0.1, "15.b does not reproduce 15.a without relief"

## 3. Cost on ERA5, suitable land, effect of the AMOC weakening: 6.b and the three versions of the floor

As 15.a. The cost on ERA5 is split between the relief cells (ΔT_z ≥ 1 °C) and the others, and north of 50° N.

In [ ]:
pct = lambda a, b: 100 * (a - b) / b if b else np.nan        # as 6.b
cost, areas, effect, regions, S6 = [], [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(B6 / f"climate_soil_scenarios_{crop}.nc") as d6:
        S = {lv: {s: np.nan_to_num(d6[f"suitable_{lv}"].sel(scenario=s).values) == 1 for s in SCEN} for lv in LEVELS}
        N = {s: d6["novel"].sel(scenario=s).values == 1 for s in SCEN}
    S6[crop] = (S, N)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * LAND_HA)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    for lv in LEVELS:
        s0 = S[lv]["ERA5"]
        for ver in VERS:
            fl = FLOOR[(crop, "ERA5", ver)]
            cost.append({"crop": crop, "level": lv, "floor": ver,
                         "suitable crop area failing the floor (points)": share(s0 & ~fl),
                         "cultivated area failing the floor (%)": share(pres & ~fl),
                         "of which relief cells (%)": share(pres & ~fl & RELIEF_CELL),
                         "of which north of 50 N (%)": share(pres & ~fl & REGIONS["north of 50 N"])})
        for s in SCEN:
            r = {"crop": crop, "scenario": s, "level": lv}
            for reg in ("global", "north of 50 N"):
                rm = REGIONS[reg]
                r[f"{reg}: suitable 6.b (Mha)"] = mha(S[lv][s] & rm)
                for ver in VERS:
                    r[f"{reg}: with the floor, {ver} (Mha)"] = mha(S[lv][s] & FLOOR[(crop, s, ver)] & rm)
            for ver in VERS:
                r[f"today's crop area suitable, floor {ver} (%)"] = share(S[lv][s] & FLOOR[(crop, s, ver)])
            areas.append(r)
        for hos, ref in PAIRS + [(run, "ERA5") for run in RUNS]:
            both = LAND & ~(N[hos] | N[ref])
            for ver in ["6.b", *VERS]:
                sh = S[lv][hos] & (FLOOR[(crop, hos, ver)] if ver != "6.b" else True)
                so = S[lv][ref] & (FLOOR[(crop, ref, ver)] if ver != "6.b" else True)
                effect.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "version": ver,
                               "suitable change, assessable in both (%)": pct(mha(sh & both), mha(so & both)),
                               "gained (Mha)": mha(sh & ~so & both), "lost (Mha)": mha(so & ~sh & both),
                               "today's crop area: change (points)": share(sh) - share(so)})
                for reg, rm in REGIONS.items():
                    ao, ah = mha(so & both & rm), mha(sh & both & rm)
                    regions.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "version": ver, "region": reg,
                                    "suitable change (%)": pct(ah, ao) if ao > 0.5 else np.nan})
COST = pd.DataFrame(cost).set_index(["crop", "level", "floor"])
AREAS = pd.DataFrame(areas).set_index(["crop", "scenario", "level"])
EFF = pd.DataFrame(effect).set_index(["crop", "pair", "level", "version"])
REG = pd.DataFrame(regions).set_index(["crop", "pair", "level", "version", "region"])
COST.to_csv(OUT / "cost_era5_lapse.csv"); AREAS.to_csv(OUT / "areas_floor_lapse.csv")
EFF.to_csv(OUT / "amoc_effect_floor_lapse.csv"); REG.to_csv(OUT / "amoc_effect_regions_floor_lapse.csv")
VORD = ["6.b", *VERS]
with pd.option_context("display.max_rows", 300, "display.max_columns", 40, "display.width", 250):
    print("(1) cost on ERA5, high input (points / % of today's crop area)")
    display(COST.xs("high", level="level").round(2))
    print("(2) suitable land, high input (Mha)")
    display(AREAS.xs("high", level="level").round(1))
    for lv in LEVELS[::-1]:
        for h, o in PAIRS[::-1]:
            t = EFF.xs((f"{h} - {o}", lv), level=("pair", "level"))["suitable change, assessable in both (%)"].unstack("version")[VORD]
            print(f"(3) AMOC effect {h[4:8]}, {lv} input, all land (%)"); display(t.round(2))
            t = REG.xs((f"{h} - {o}", lv, "north of 50 N"), level=("pair", "level", "region"))["suitable change (%)"].unstack("version")[VORD]
            print("    north of 50 N (%)"); display(t.round(2))
            g = EFF.xs((f"{h} - {o}", lv), level=("pair", "level"))
            t = (g["gained (Mha)"] - g["lost (Mha)"]).unstack("version")[VORD]
            print("    net gained - lost, all land (Mha)"); display(t.round(0))

## 4. Figures

(a) Warming of the lower part of the cell ΔT_z, and per crop the cultivated cells (MIRCA2000) failing the floor on ERA5:
recovered by the correction (blue), recovered only by not applying the floor in relief cells (purple), failing every
version (red). (b) Effect of the AMOC weakening in 2050: 6.b and the three versions of the floor.

In [ ]:
LV = "high"
fig, axes = plt.subplots(len(CROPS) + 1, 1, figsize=(11, 4.2 * (len(CROPS) + 1)), squeeze=False)
ax = axes[0, 0]
im = ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, DZ_GRID, np.nan), cmap="viridis", vmin=0, vmax=6, shading="nearest")
ax.set_ylim(-60, 85); fig.colorbar(im, ax=ax, label="degC")
ax.set_title("warming of the lower part of the cell against its mean: 0.0065 x (mean - 10th percentile of the elevation)", fontsize=9)
for ax, crop in zip(axes[1:, 0], CROPS):
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * LAND_HA)
    f0, f1, f2 = (FLOOR[(crop, "ERA5", v)] for v in VERS)
    c = np.where(~LAND, np.nan, np.where(~pres, 0.0, np.where(f0, 1.0, np.where(f1, 2.0, np.where(f2, 3.0, 4.0)))))
    ax.pcolormesh(GRID["lon"], GRID["lat"], c,
                  cmap=plt.matplotlib.colors.ListedColormap(["#eeeeee", "#a6d96a", "#2c7bb6", "#984ea3", "#d7191c"]),
                  vmin=-0.5, vmax=4.5, shading="nearest")
    ax.set_ylim(-60, 85)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    ax.set_title(f"{crop}, cultivated on ERA5: passes on the cell mean (green); recovered by the correction (blue) "
                 f"{share(pres & ~f0 & f1):.1f} %; only by not applying in relief cells (purple) {share(pres & ~f1 & f2):.1f} %; "
                 f"fails all (red) {share(pres & ~f1 & ~f2):.1f} %", fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "recovered_by_lapse_era5.png", dpi=70); plt.close(fig)

h, o = PAIRS[-1]
COLS = {"6.b": "#4d4d4d", VERS[0]: "#d95f02", VERS[1]: "#7570b3", VERS[2]: "#1b9e77"}
fig, axes = plt.subplots(1, 2, figsize=(14, 0.8 * len(CROPS) + 2))
for ax, (src, title) in zip(axes, [("global", "all land (assessable in both)"), ("north of 50 N", "north of 50 N")]):
    for n, ver in enumerate(VORD):
        if src == "global":
            vals = [EFF.loc[(c, f"{h} - {o}", LV, ver), "suitable change, assessable in both (%)"] for c in CROPS]
        else:
            vals = [REG.loc[(c, f"{h} - {o}", LV, ver, src), "suitable change (%)"] for c in CROPS]
        ax.barh(np.arange(len(CROPS)) + (n - 1.5) * 0.21, vals, height=0.2, color=COLS[ver],
                label="6.b" if ver == "6.b" else f"with the heat floor, {ver}")
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(CROPS)))
    ax.set_yticklabels([c.replace("_rf", "") for c in CROPS]); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
    ax.set_title(f"{title}: suitable land change (%)", fontsize=10)
axes[0].legend(fontsize=8)
fig.suptitle(f"Effect of the AMOC weakening (2050 hosing - no hosing, {LV} input): 6.b and three versions of the heat-unit floor", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"amoc_effect_floor_lapse_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))

## 5. Reading

To be written after the run on BDAP: (1) how much of the cost of the floor on ERA5 each version removes, in the relief
cells and north of 50° N; (2) whether the land removed north of 50° N (flat) stays as in 15.a; (3) whether the effect
of the AMOC weakening with the floor holds in every version; (4) which version, if any, can join the climate module
next to TMIN.